# ORAC-NT Reproducibility Package
## Step-by-step reproduction of 5/5 GW event detection

**Author:** Dimitar Kretski  
**DOI:** 10.5281/zenodo.20129975  
**GitHub:** github.com/Kretski/ORAC-NT-Astrophysics  

This notebook reproduces the core ORAC-NT detection results on 5 known GW events.  
All data is publicly available from GWOSC (gwosc.org).

### Requirements
```
pip install gwosc gwpy pycbc numpy scipy matplotlib
```

### Events covered
| Event | GPS Start | GPS End | Detector | Type |
|-------|-----------|---------|----------|------|
| GW150914 | 1126259462 | 1126259562 | H1 | BBH |
| GW170814 | 1186741861 | 1186741961 | H1 | BBH |
| GW170817 | 1187008882 | 1187008982 | H1 | BNS |
| GW200105 | 1262276512 | 1262276612 | H1 | NSBH |
| GW200115 | 1263089122 | 1263089222 | H1 | NSBH |


## 1. Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import kurtosis
import warnings
warnings.filterwarnings('ignore')

# GWOSC data access
try:
    from gwosc.datasets import event_gps
    from gwpy.timeseries import TimeSeries
    HAS_GWPY = True
    print("gwpy available - will fetch real GWOSC data")
except ImportError:
    HAS_GWPY = False
    print("gwpy not available - will use simulated data for demo")

print("ORAC-NT Reproducibility Package ready")


## 2. Event GPS intervals (exact)

In [ ]:
# Exact GPS intervals used in ORAC-NT Phase E validation
EVENTS = {
    'GW150914': {
        'gps':      1126259462.4,
        'gps_start': 1126259412,   # -50s before merger
        'gps_end':   1126259512,   # +50s after merger
        'detector': 'H1',
        'type':     'BBH',
        'expected_snr': '>10',
        'channel': 'H1:GWOSC-16KHZ_R1_STRAIN',
    },
    'GW170814': {
        'gps':      1186741861.5,
        'gps_start': 1186741811,
        'gps_end':   1186741911,
        'detector': 'H1',
        'type':     'BBH',
        'expected_snr': '>10',
        'channel': 'H1:GWOSC-16KHZ_R1_STRAIN',
    },
    'GW170817': {
        'gps':      1187008882.4,
        'gps_start': 1187008832,
        'gps_end':   1187008932,
        'detector': 'H1',
        'type':     'BNS',
        'expected_snr': '>8',
        'channel': 'H1:GWOSC-16KHZ_R1_STRAIN',
    },
    'GW200105': {
        'gps':      1262276512.0,
        'gps_start': 1262276462,
        'gps_end':   1262276562,
        'detector': 'H1',
        'type':     'NSBH',
        'expected_snr': '>10',
        'channel': 'H1:GWOSC-16KHZ_R1_STRAIN',
    },
    'GW200115': {
        'gps':      1263089122.0,
        'gps_start': 1263089072,
        'gps_end':   1263089172,
        'detector': 'H1',
        'type':     'NSBH',
        'expected_snr': '>10',
        'channel': 'H1:GWOSC-16KHZ_R1_STRAIN',
    },
}

print("Events loaded:")
for name, ev in EVENTS.items():
    print(f"  {name}: GPS={ev['gps']} | {ev['type']} | {ev['detector']}")


## 3. Preprocessing pipeline

Exact preprocessing steps used in ORAC-NT:
1. Load strain data (16kHz)
2. Bandpass filter 20-500Hz (Butterworth 4th order)
3. Welch PSD estimation (4s segments)
4. PSD whitening
5. Tukey window (alpha=0.05)


In [ ]:
FS = 16384  # Hz - GWOSC 16kHz channel
F_LOW  = 20    # Hz
F_HIGH = 500   # Hz

def bandpass(data, fs=FS, f_low=F_LOW, f_high=F_HIGH):
    """Butterworth bandpass 20-500Hz"""
    sos = signal.butter(4, [f_low, f_high],
                        btype='bandpass', fs=fs, output='sos')
    return signal.sosfiltfilt(sos, data)

def estimate_psd(data, fs=FS, seg_duration=4.0):
    """Welch PSD estimation - 4s segments"""
    nperseg = int(seg_duration * fs)
    f, psd = signal.welch(data, fs, nperseg=nperseg)
    return f, psd

def whiten(data, fs=FS, seg_duration=4.0):
    """PSD whitening"""
    f, psd = estimate_psd(data, fs, seg_duration)
    freqs  = np.fft.rfftfreq(len(data), 1.0/fs)
    psd_i  = np.interp(freqs, f, psd)
    white  = np.fft.irfft(
        np.fft.rfft(data) / np.sqrt(psd_i + 1e-30),
        n=len(data)
    )
    window = signal.tukey(len(white), alpha=0.05)
    return np.nan_to_num(white) * window

def preprocess(data, fs=FS):
    """Full preprocessing pipeline"""
    bp = bandpass(data, fs)
    wh = whiten(bp, fs)
    cal_std = np.std(wh[:int(fs * 2)])
    return wh / (cal_std + 1e-12)

print("Preprocessing pipeline defined")
print(f"  Bandpass: {F_LOW}-{F_HIGH} Hz")
print(f"  PSD: Welch, 4s segments")
print(f"  Whitening: PSD division in frequency domain")
print(f"  Window: Tukey alpha=0.05")


## 4. ORAC-NT Detection Algorithm

Two-channel detector:
- **RMS channel**: rolling RMS energy
- **ORAC H channel**: H = E × S (energy × spectral entropy)
- **Chi-squared veto**: sub-band consistency
- **Fisher kurtosis veto**: non-Gaussian glitch rejection
- **MAD-based threshold**: absolute sigma threshold

Detection criterion: SNR > 5σ in either channel


In [ ]:
def spectral_entropy(data, fs=FS, nperseg=512):
    """Spectral entropy S = -sum(p*log(p))"""
    _, _, Sxx = signal.spectrogram(data, fs, nperseg=nperseg)
    psd_norm = Sxx / (np.sum(Sxx, axis=0) + 1e-20)
    entropy  = -np.sum(psd_norm * np.log(psd_norm + 1e-20), axis=0)
    return entropy

def orac_h_metric(data, fs=FS, window_s=0.1):
    """ORAC H = E * S (energy * spectral entropy)"""
    win   = int(window_s * fs)
    rms   = np.array([
        np.sqrt(np.mean(data[i:i+win]**2))
        for i in range(0, len(data)-win, win//2)
    ])
    ent   = spectral_entropy(data, fs)
    # Interpolate entropy to match rms length
    ent_i = np.interp(
        np.linspace(0, 1, len(rms)),
        np.linspace(0, 1, len(ent)),
        ent
    )
    return rms * ent_i, rms

def kurtosis_veto(data, fs=FS, window_s=1.0, threshold=10.0):
    """Fisher kurtosis veto - flags non-Gaussian segments"""
    win   = int(window_s * fs)
    kurts = []
    for i in range(0, len(data)-win, win):
        k = kurtosis(data[i:i+win], fisher=True, bias=False)
        kurts.append(k)
    return np.array(kurts), np.any(np.abs(kurts) > threshold)

def chi2_veto(data, fs=FS, n_bands=8):
    """Chi-squared sub-band veto"""
    band_size = (F_HIGH - F_LOW) // n_bands
    band_snrs = []
    for i in range(n_bands):
        f_lo = F_LOW + i * band_size
        f_hi = F_LOW + (i+1) * band_size
        sos  = signal.butter(4, [f_lo, f_hi],
                             btype='bandpass', fs=fs, output='sos')
        band = signal.sosfiltfilt(sos, data)
        band_snrs.append(np.max(np.abs(band)))
    chi2 = np.std(band_snrs) / (np.mean(band_snrs) + 1e-12)
    return chi2, chi2 < 0.5  # passes if consistent

def detect_orac(data, fs=FS, sigma_threshold=5.0):
    """
    Full ORAC-NT detection pipeline.
    Returns: detected (bool), SNR, trigger_time_offset
    """
    # Preprocess
    proc = preprocess(data, fs)

    # ORAC H metric + RMS
    h_metric, rms = orac_h_metric(proc, fs)

    # MAD-based threshold (absolute sigma)
    mad_h   = np.median(np.abs(h_metric - np.median(h_metric)))
    mad_rms = np.median(np.abs(rms - np.median(rms)))

    threshold_h   = np.median(h_metric)   + sigma_threshold * 1.4826 * mad_h
    threshold_rms = np.median(rms)        + sigma_threshold * 1.4826 * mad_rms

    # Peak detection
    peak_h   = np.max(h_metric)
    peak_rms = np.max(rms)

    snr_h   = (peak_h   - np.median(h_metric))   / (1.4826 * mad_h   + 1e-12)
    snr_rms = (peak_rms - np.median(rms))         / (1.4826 * mad_rms + 1e-12)
    snr     = max(snr_h, snr_rms)

    # Veto checks
    _, kurt_flag = kurtosis_veto(proc, fs)
    _, chi2_pass = chi2_veto(proc, fs)

    detected = (
        (peak_h > threshold_h or peak_rms > threshold_rms)
        and not kurt_flag
        and chi2_pass
    )

    # Trigger time offset
    win_s  = 0.1
    win    = int(win_s * fs)
    t_peak = np.argmax(h_metric) * (win // 2) / fs

    return detected, snr, t_peak

print("ORAC-NT detector defined")
print("  Channels: RMS + ORAC H (E*S)")
print("  Vetos: Fisher kurtosis + Chi-squared sub-band")
print("  Threshold: MAD-based absolute sigma")


## 5. Detection demo

This cell demonstrates ORAC-NT on simulated GW-like signals.  
For real GWOSC data, uncomment the `gwpy` section below.


In [ ]:
# ── Simulated demo (no GWOSC credentials needed) ──────────────
duration = 4.0  # seconds
t = np.linspace(0, duration, int(FS * duration))

results = {}

for name, ev in EVENTS.items():

    # Simulate noise + chirp signal
    noise = np.random.randn(len(t)) * 1e-21

    # Simple chirp proxy
    f0, f1 = 30, 300
    chirp_t = t[len(t)//2 - int(0.2*FS) : len(t)//2 + int(0.1*FS)]
    chirp   = np.zeros(len(t))

    if ev['type'] == 'BNS':
        amp = 3e-22
    else:
        amp = 8e-22

    chirp[len(t)//2 - int(0.2*FS):
          len(t)//2 + int(0.1*FS)] = (
        amp * np.sin(2*np.pi*np.linspace(f0, f1, len(chirp_t))*chirp_t)
    )

    strain = noise + chirp

    # Run ORAC-NT
    detected, snr, t_trig = detect_orac(strain, FS)

    results[name] = {
        'detected': detected,
        'snr':      snr,
        't_trigger': t_trig,
        'type':     ev['type'],
    }

    status = "DETECTED" if detected else "MISSED"
    print(f"  {name} ({ev['type']}): {status} | SNR={snr:.2f} | t={t_trig:.3f}s")

n_detected = sum(1 for r in results.values() if r['detected'])
print(f"\nResult: {n_detected}/5 detected")
print("Note: This uses simulated data. Real GWOSC results: 5/5")


## 6. Real GWOSC data (requires gwpy)

Uncomment and run with real GWOSC data:


In [ ]:
# ── Real GWOSC data ───────────────────────────────────────────
# Uncomment to run with real data

# if HAS_GWPY:
#     for name, ev in EVENTS.items():
#         print(f"Fetching {name}...")
#         strain = TimeSeries.fetch_open_data(
#             ev['detector'],
#             ev['gps_start'],
#             ev['gps_end'],
#             sample_rate=FS,
#             cache=True
#         )
#         data = strain.value
#         detected, snr, t_trig = detect_orac(data, FS)
#         merger_gps = ev['gps']
#         t_offset = (ev['gps_start'] + t_trig) - merger_gps
#         print(f"  {name}: {'DETECTED' if detected else 'MISSED'}")
#         print(f"  SNR={snr:.2f} | offset={t_offset:+.3f}s")

print("Uncomment above to run on real GWOSC data")
print("Data will be cached locally after first download")


## 7. Visualization

In [ ]:
# Plot detection summary
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(results.keys())
snrs  = [results[n]['snr'] for n in names]
types = [results[n]['type'] for n in names]
cols  = ['#00ffcc' if results[n]['detected'] else '#ff4444' for n in names]

axes[0].bar(names, snrs, color=cols)
axes[0].axhline(5.0, color='red', ls='--', label='5σ threshold')
axes[0].set_title('ORAC-NT SNR per Event (Simulated)')
axes[0].set_ylabel('SNR (sigma)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

detected_count = sum(1 for r in results.values() if r['detected'])
axes[1].pie(
    [detected_count, 5-detected_count],
    labels=['Detected', 'Missed'],
    colors=['#00ffcc', '#ff4444'],
    autopct='%1.0f%%',
    startangle=90
)
axes[1].set_title('Detection Rate (Simulated)')

plt.suptitle(
    'ORAC-NT Detection Results\n'
    'doi.org/10.5281/zenodo.20129975 | Dimitar Kretski',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('orac_nt_reproducibility.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: orac_nt_reproducibility.png")


## 8. Summary

### Reproduction steps
1. Install requirements: `pip install gwosc gwpy pycbc numpy scipy matplotlib`
2. Run cells 1-5 for simulated demo
3. Uncomment cell 6 for real GWOSC data
4. Results should match: 5/5 detection on known events

### Key parameters
| Parameter | Value |
|-----------|-------|
| Sample rate | 16384 Hz |
| Bandpass | 20-500 Hz |
| PSD method | Welch, 4s segments |
| Threshold | 5σ MAD-based |
| Window | Tukey α=0.05 |

### Expected results (real GWOSC data)
| Event | SNR | Detected | Channel |
|-------|-----|----------|---------|
| GW150914 | >10 | YES | RMS |
| GW170814 | >10 | YES | RMS |
| GW170817 | >8  | YES | ORAC H |
| GW200105 | >10 | YES | RMS |
| GW200115 | >10 | YES | RMS |

### References
- Zenodo: doi.org/10.5281/zenodo.20129975
- GitHub: github.com/Kretski/ORAC-NT-Astrophysics
- ESSOAr: doi.org/10.22541/essoar.15003714/v1
